In [ ]:
# =========================
# FINAL: Enhanced Document Q&A (Fast + Submit-ready)
# - OCR fallback (optional)
# - Metadata tagging
# - Embeddings + FAISS
# - Routing + Answer using Mistral API (NO Gemini, NO local 7B load)
# - Gradio UI: upload, chat, sources, confidence, chunks used, save history
# =========================

# -------- Install (Colab) --------
!pip -q install gradio PyMuPDF PyPDF2 sentence-transformers faiss-cpu mistralai
# OCR optional (only if you need scanned PDFs)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!pip -q install pytesseract pillow

import os, re, json, hashlib
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import gradio as gr

from PIL import Image
import pytesseract

from mistralai.client import Mistral

# =========================
# 0) CONFIG
# =========================
# Put your key here OR set it in Colab: os.environ["MISTRAL_API_KEY"]="..."
import os
os.environ["MISTRAL_API_KEY"] = "YOUR_NEW_KEY"


MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "").strip()
if not MISTRAL_API_KEY:
    print("⚠️ Set MISTRAL_API_KEY first: os.environ['MISTRAL_API_KEY']='YOUR_KEY'")

MISTRAL_MODEL = "mistral-small-latest"  # fast & stable
client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

# Speed knobs:
MAX_LLM_PAGES_FOR_BOUNDARY = 8     # only do boundary LLM checks for first N pages
CHUNK_SIZE_CHARS = 1200           # chunk size for retrieval
CHUNK_OVERLAP_CHARS = 200
TOP_K = 4                         # chunks to retrieve per query

ENABLE_OCR_FALLBACK = True        # if scanned PDFs; set False to speed up
OCR_DPI = 200                     # higher -> slower, more accurate


# =========================
# 1) UTILITIES
# =========================
def clean_text(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def sliding_chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    text = text or ""
    chunks = []
    i = 0
    n = len(text)
    while i < n:
        chunk = text[i:i+chunk_size]
        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)
        i += max(1, chunk_size - overlap)
    return chunks

def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:12]


# =========================
# 2) MISTRAL API CALL (single function used everywhere)
# =========================
def mistral_generate(prompt: str, max_tokens: int = 256) -> str:
    if client is None:
        raise RuntimeError("MISTRAL_API_KEY not set.")
    res = client.chat.complete(
        model=MISTRAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return (res.choices[0].message.content or "").strip()


# =========================
# 3) OCR (only used if needed)
# =========================
def ocr_page(pymupdf_page: fitz.Page) -> str:
    # Render page to image then OCR
    mat = fitz.Matrix(OCR_DPI/72, OCR_DPI/72)
    pix = pymupdf_page.get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img)
    return clean_text(text)


# =========================
# 4) DOCUMENT INGESTION
# =========================
@dataclass
class PageItem:
    page_index: int
    text: str

def extract_pages_from_pdf(pdf_path: str) -> List[PageItem]:
    doc = fitz.open(pdf_path)
    pages: List[PageItem] = []
    for i in range(len(doc)):
        page = doc[i]
        txt = clean_text(page.get_text("text"))
        # OCR fallback if no text
        if ENABLE_OCR_FALLBACK and len(txt) < 10:
            try:
                txt = ocr_page(page)
            except Exception:
                txt = txt  # keep empty if OCR fails
        pages.append(PageItem(page_index=i, text=txt))
    doc.close()
    return pages


# =========================
# 5) ROUTING / CLASSIFICATION / BOUNDARY (LLM, but minimized)
# =========================
DOC_TYPES = [
    "Loan Estimate",
    "Closing Disclosure",
    "Lender Fee Sheet",
    "Mortgage Statement",
    "Contract",
    "PaySlip",
    "ID",
    "Resume",
    "Other"
]

def classify_document_type(page_text: str) -> str:
    sample = (page_text or "")[:1200]
    prompt = f"""
You are classifying a document page into ONE type.

Choose ONE from: {", ".join(DOC_TYPES)}.

Page text:
\"\"\"{sample}\"\"\"

Return ONLY the label, nothing else.
"""
    out = mistral_generate(prompt, max_tokens=12)
    out = re.sub(r"[^A-Za-z ]", "", out).strip().title()
    return out if out in DOC_TYPES else "Other"

def is_same_document(prev_text: str, curr_text: str, current_doc_type: Optional[str]) -> bool:
    prev_sample = (prev_text or "")[-600:]
    curr_sample = (curr_text or "")[:600]
    prompt = f"""
Determine if these two pages belong to the SAME document.

Current doc type (if known): {current_doc_type or "Unknown"}

End of previous page:
\"\"\"{prev_sample}\"\"\"

Start of current page:
\"\"\"{curr_sample}\"\"\"

Answer ONLY Yes or No.
"""
    out = mistral_generate(prompt, max_tokens=6).lower()
    return out.startswith("y")  # Yes -> True, No -> False

def predict_doc_type_for_query(query: str) -> str:
    q = query.lower()

    # HARD ROUTING RULES (fast + reliable)
    if any(x in q for x in ["monthly payment", "estimated payment", "principal and interest", "p&i", "mortgage payment"]):
        return "Loan Estimate"
    if any(x in q for x in ["lender fee", "origination", "underwriting", "processing fee", "points"]):
        return "Lender Fee Sheet"
    if any(x in q for x in ["closing disclosure", "cash to close"]):
        return "Closing Disclosure"
    prompt = f"""
You route user questions to the most relevant document type.

Choose ONE from: {", ".join(DOC_TYPES)}.

User question: "{query}"

Return ONLY the label.
"""
    out = mistral_generate(prompt, max_tokens=10)
    out = re.sub(r"[^A-Za-z ]", "", out).strip().title()
    return out if out in DOC_TYPES else "Other"


# =========================
# 6) BUILD CHUNKS + METADATA STORE + FAISS INDEX
# =========================
# We store chunks with metadata; FAISS stores embeddings.
chunk_store: List[Dict[str, Any]] = []
faiss_index: Optional[faiss.IndexFlatIP] = None  # cosine via normalized dot
doc_summary: Dict[str, Any] = {}

def reset_store():
    global chunk_store, faiss_index, doc_summary
    chunk_store = []
    faiss_index = None
    doc_summary = {}

def build_index_from_pdfs(pdf_files: List[str]) -> str:
    """
    Ingest PDFs -> page extraction -> (optional) boundary grouping -> chunking -> embeddings -> FAISS
    """
    global chunk_store, faiss_index, doc_summary
    reset_store()

    all_chunks_text: List[str] = []
    all_chunk_meta: List[Dict[str, Any]] = []

    total_pages = 0
    total_docs_detected = 0

    for pdf_path in pdf_files:
        source_file = os.path.basename(pdf_path)
        pages = extract_pages_from_pdf(pdf_path)
        total_pages += len(pages)

        # --- Detect logical documents inside this PDF (minimize LLM calls) ---
        current_doc_id = f"{source_file}::doc0"
        current_doc_type = classify_document_type(pages[0].text if pages else "") if pages else "Other"
        total_docs_detected += 1
        page_in_doc = 0

        for i, p in enumerate(pages):
            if i == 0:
                pass
            else:
                # only boundary-check for first N pages to avoid huge latency
                if i <= MAX_LLM_PAGES_FOR_BOUNDARY:
                    same = is_same_document(pages[i-1].text, p.text, current_doc_type)
                else:
                    same = True

                if not same:
                    # new logical doc starts here
                    total_docs_detected += 1
                    current_doc_id = f"{source_file}::doc{total_docs_detected-1}"
                    current_doc_type = classify_document_type(p.text)
                    page_in_doc = 0
                else:
                    page_in_doc += 1

            # --- Chunk this page (page-level chunks) ---
            page_text = p.text or ""
            chunks = sliding_chunk_text(page_text, CHUNK_SIZE_CHARS, CHUNK_OVERLAP_CHARS)
            for ci, ch in enumerate(chunks):
                meta = {
                    "source_file": source_file,
                    "doc_id": current_doc_id,
                    "doc_type": current_doc_type,
                    "page": p.page_index,
                    "page_in_doc": page_in_doc,
                    "chunk_index": ci,
                    "chunk_id": f"{sha1(source_file)}-{p.page_index}-{ci}"
                }
                all_chunks_text.append(ch)
                all_chunk_meta.append(meta)

    if not all_chunks_text:
        return "No text extracted. Try enabling OCR or use a text-based PDF."

    # --- Embeddings + FAISS (cosine) ---
    embs = embedder.encode(all_chunks_text, normalize_embeddings=True)
    embs = np.asarray(embs, dtype="float32")

    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)

    chunk_store = [{"text": t, "metadata": m} for t, m in zip(all_chunks_text, all_chunk_meta)]
    faiss_index = idx

    doc_summary = {
        "pdf_count": len(pdf_files),
        "total_pages": total_pages,
        "logical_docs_detected": total_docs_detected,
        "total_chunks": len(chunk_store),
        "embed_model": EMBED_MODEL_NAME,
    }

    return f"✅ Processed {len(pdf_files)} PDF(s). Pages: {total_pages}. Chunks: {len(chunk_store)}. Logical docs: {total_docs_detected}."


# =========================
# 7) RETRIEVE + ANSWER (with sources + confidence)
# =========================
def retrieve_chunks(query: str, k: int, doc_type_filter: str):
    if faiss_index is None:
        return [], 0.0

    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    D, I = faiss_index.search(q_emb, min(max(k * 5, 10), len(chunk_store)))
    D = D[0].tolist()
    I = I[0].tolist()

    candidates = []
    scores = []

    for score, idx in zip(D, I):
        item = chunk_store[idx]
        if doc_type_filter != "All" and item["metadata"]["doc_type"] != doc_type_filter:
            continue
        candidates.append(item)
        scores.append(float(score))
        if len(candidates) >= k:
            break

    conf = 0.0
    if scores:
        conf = float(np.clip(np.mean(scores), 0.0, 1.0))

    # ✅ fallback: if filter gives nothing, try again with All
    if len(candidates) == 0 and doc_type_filter != "All":
        return retrieve_chunks(query, k, "All")

    return candidates, conf


def build_answer_prompt(query: str, retrieved: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for r in retrieved:
        md = r["metadata"]
        cite = f"[{md['source_file']} | {md['doc_type']} | page {md['page']}]"
        context_blocks.append(f"{cite}\n{r['text']}")
    context = "\n\n---\n\n".join(context_blocks)

    prompt = f"""
You are a careful assistant answering questions about mortgage-related documents.

Rules:
- Use ONLY the context below. Do not guess.
- If the answer is not explicitly in the context, output exactly: Not found
- When you provide a number/value, you MUST quote the exact line from the context that contains it.

User question:
{query}

Context:
{context}

Output format (follow exactly):
Answer: <one short sentence with the value if present>
Evidence: "<quote the exact line that contains the value>"
Sources: (Source: <filename>, <doc_type>, page <page>)
"""
    return prompt

def answer_query(query: str, manual_doc_type: str, auto_route: bool, k: int) -> Dict[str, Any]:
    # decide doc_type
    if auto_route:
        predicted = predict_doc_type_for_query(query)
        doc_type_filter = predicted if predicted in DOC_TYPES else "Other"
    else:
        predicted = manual_doc_type if manual_doc_type in DOC_TYPES else "Other"
        doc_type_filter = manual_doc_type

    if manual_doc_type == "All" and not auto_route:
        predicted = "All"
        doc_type_filter = "All"

    retrieved, conf = retrieve_chunks(query, k=k, doc_type_filter=doc_type_filter)

    if not retrieved:
        return {
            "query": query,
            "predicted_doc_type": predicted,
            "chunks_used": 0,
            "confidence": 0.0,
            "sources": [],
            "answer": "No relevant chunks were retrieved. Try changing doc_type filter or uploading a different PDF."
        }

    prompt = build_answer_prompt(query, retrieved)
    ans = mistral_generate(prompt, max_tokens=220)

    sources = []
    for r in retrieved:
        md = r["metadata"]
        sources.append({
            "source_file": md["source_file"],
            "doc_type": md["doc_type"],
            "page": md["page"],
            "chunk_id": md["chunk_id"],
        })

    return {
        "query": query,
        "predicted_doc_type": predicted,
        "chunks_used": len(retrieved),
        "confidence": round(conf, 3),
        "sources": sources,
        "answer": ans
    }


# =========================
# 8) GRADIO UI
# =========================
def ui_process(files):
    if files is None or len(files) == 0:
        return "Please upload at least 1 PDF.", json.dumps({}, indent=2)

    pdf_paths = []
    for f in files:
        pdf_paths.append(f.name)

    status = build_index_from_pdfs(pdf_paths)
    return status, json.dumps(doc_summary, indent=2)

def ui_ask(question, history, doc_type_filter, auto_route, k):
    if history is None:
        history = []

    if not question or not question.strip():
        return history, "{}", ""

    if faiss_index is None:
        history = history + [{"role": "assistant", "content": "Please upload and process PDFs first."}]
        return history, "{}", ""

    result = answer_query(
        query=question.strip(),
        manual_doc_type=doc_type_filter,
        auto_route=auto_route,
        k=int(k)
    )

    # chat display
    history = history + [
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": result["answer"]}
    ]

    # pretty json outputs
    result_json = json.dumps(result, indent=2)
    sources_text = "\n".join([f"- {s['source_file']} | {s['doc_type']} | page {s['page']} | chunk {s['chunk_id']}" for s in result["sources"]])

    return history, result_json, sources_text

def ui_save_chat(history):
    if history is None:
        history = []
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"chat_history_{ts}.txt"
    with open(path, "w", encoding="utf-8") as f:
        for m in history:
            f.write(f"{m['role'].upper()}: {m['content']}\n\n")
    return f"Saved: {path}"

with gr.Blocks(title="Enhanced Document Q&A") as demo:
    gr.Markdown("## 🚀 Enhanced Document Q&A System (Fast RAG + Mistral API)")
    gr.Markdown("Upload mortgage-related PDFs, then ask questions. The system returns an answer with sources + confidence.")

    with gr.Row():
        files = gr.Files(label="Upload PDF(s)", file_types=[".pdf"])
        process_btn = gr.Button("Process Document(s)")

    status = gr.Textbox(label="Status")
    docinfo = gr.Textbox(label="Document Info (debug summary)", lines=6)

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", type="messages")
            question = gr.Textbox(label="Ask a question", placeholder="e.g., What is the total lender fee?")
            ask_btn = gr.Button("Ask")
            save_btn = gr.Button("Save Chat History")
            save_status = gr.Textbox(label="Save Status")
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")
            doc_type_filter = gr.Dropdown(
                choices=["All"] + DOC_TYPES,
                value="All",
                label="Document Type Filter"
            )
            auto_route = gr.Checkbox(value=True, label="Auto-Route Queries (LLM)")
            k_slider = gr.Slider(1, 8, value=TOP_K, step=1, label="Chunks to Retrieve")

            result_json = gr.Textbox(label="Result JSON (for submission)", lines=14)
            sources_box = gr.Textbox(label="Sources", lines=8)

    process_btn.click(ui_process, inputs=[files], outputs=[status, docinfo])
    ask_btn.click(ui_ask, inputs=[question, chatbot, doc_type_filter, auto_route, k_slider],
                 outputs=[chatbot, result_json, sources_box])
    save_btn.click(ui_save_chat, inputs=[chatbot], outputs=[save_status])

demo.launch(share=True, debug=True)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_38465/256004942.py:493: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", type="messages")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://94e8ae6b001aa1d9d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
